In [1]:
import pyfeyngym
import ppo_masking
import gymnasium as gym
import torch
from torch import nn

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:
# set matplotlib tk backend if possible, otherwise inline backend
from IPython import get_ipython
ip = get_ipython()
try:
    ip.run_line_magic("matplotlib", "tk")
except Exception:
    ip.run_line_magic("matplotlib", "inline")

In [3]:
max_seed_propgator_power = 6
board_size = max_seed_propgator_power + 3 # -1 to (max_seed_propgator_power+1)
eval_target_integral = [6, 6] # target integral used to evaluate training outcome, even though the training is for arbitrary integrals

In [4]:
assert board_size % 3 == 0 # Need to board size to a multiple of 3, for our CNN setup

In [5]:
def make_critic():
    network = nn.Sequential(
        nn.Conv2d(pyfeyngym.N_CHANNELS, 128, kernel_size=3, stride=1, padding=1),
        nn.GELU(),
        nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
        nn.GELU(),
        nn.Conv2d(256, 8, kernel_size=3, stride=3, padding=0),
        nn.GELU(),
        nn.Flatten(),
        ppo_masking.layer_init(nn.Linear(8*int(board_size/3)**2, 1), std=1.0)
    )
    return network

def make_actor():
    network = nn.Sequential(
        nn.Conv2d(pyfeyngym.N_CHANNELS, 128, kernel_size=3, stride=1, padding=1),
        nn.GELU(),
        nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
        nn.GELU(),
        nn.Conv2d(256, 16, kernel_size=3, stride=1, padding=1),
        nn.GELU(),
        ppo_masking.layer_init(nn.Conv2d(16, 3, kernel_size=3, stride=1, padding=1), std=0.01),
        nn.Flatten()
    )
    return network

In [6]:
actor = make_actor()
critic = make_critic()
args = ppo_masking.Args(env_id = "pyfeyngym-v0", gamma=0.999, actor=actor, critic=critic,
                        num_steps=4096, total_timesteps=2400000,
                        learning_rate = 3e-4, anneal_lr = False, seed=1,
                        env_kwargs = {
                            "random_target": True,
                            "max_seed_propagator_power": max_seed_propgator_power
                        },
                        eval_env_kwargs = {
                            "random_target": False,
                            "target_integral": eval_target_integral,
                            "max_seed_propagator_power": max_seed_propgator_power
                        }
                       )

In [8]:
%%time
trained_agent = ppo_masking.run(args)
torch.save(actor, "multitarget_actor.pth")
torch.save(critic, "multitarget_critic.pth")

[6, 6]
eval_result = -195.25000000000006
SPS: 584
[6, 6]
eval_result = -122.3055555555556
SPS: 648
[6, 6]
eval_result = -120.61111111111114
SPS: 681
[6, 6]
eval_result = -105.61111111111113
SPS: 695
[6, 6]
eval_result = -96.47222222222224
SPS: 702
[6, 6]
eval_result = -109.47222222222221
SPS: 706
[6, 6]
eval_result = -79.05555555555559
SPS: 702
[6, 6]
eval_result = -87.16666666666669
SPS: 706
[6, 6]
eval_result = -75.91666666666666
SPS: 706
[6, 6]
eval_result = -82.72222222222221
SPS: 710
[6, 6]
eval_result = -89.38888888888891
SPS: 713
[6, 6]
eval_result = -55.722222222222214
SPS: 715
[6, 6]
eval_result = -53.83333333333333
SPS: 717
[6, 6]
eval_result = -63.05555555555555
SPS: 719
[6, 6]
eval_result = -60.72222222222224
SPS: 721
[6, 6]
eval_result = -49.41666666666666
SPS: 721
[6, 6]
eval_result = -66.47222222222221
SPS: 722
[6, 6]
eval_result = -48.305555555555536
SPS: 723
[6, 6]
eval_result = -37.58333333333332
SPS: 724
[6, 6]
eval_result = -39.80555555555555
SPS: 725
[6, 6]
eval_re

In [7]:
# For loading the previously trained actor network, skip the previous cell
# and run this cell instead
actor = torch.load("multitarget_actor.pth", weights_only=False)

In [8]:
def test_env(target_integral):
    total_reward, finished_env = ppo_masking.test_total_reward("pyfeyngym-v0", actor, return_finished_env=True,
                                                           env_kwargs = {"target_integral": target_integral,
                                                                        "random_target": False,
                                                                        "normalize_reward": False})
    fig, ax, anim = pyfeyngym.visualize_board(finished_env.unwrapped, finished_env.unwrapped.state.actual_target_integral, (board_size, board_size))
    fig.savefig(f"figs/feyngym_multitarget_{target_integral[0]}_{target_integral[1]}.png")
    anim.save(f"figs/feyngym_multitarget_{target_integral[0]}_{target_integral[1]}.gif")
    return total_reward

In [10]:
test_env([6,6])

-182.0

In [11]:
test_env([6,5])

-162.0

In [9]:
test_env([3,3])

-74.0